# 18. The stack, with CatBoost as a nineteenth member

**One variable against ledger row 25** (`stack_logit_18_oof`, CV 0.967650): the same
logistic combiner, the same `C`, the same fold-wise fitting protocol, the same five
folds. The member set goes from eighteen to nineteen.

Row 25 is refit here from the same vectors rather than quoted from the ledger, so the
comparison is paired per fold inside one run rather than across two.

## Why the combiner is fit inside the fold loop

For each outer fold the logistic regression is fit on the other four folds of the
out-of-fold matrix and scored on the held-out one. The member vectors are already
out-of-fold, so a row's member predictions come from models that never saw it and its
combiner weights come from a fit that never saw it either. That is what makes this CV
comparable to every other row in the file, which row 24's in-sample number was not.

## What CatBoost is expected to be worth

The probe in `17_catboost_te.ipynb` measured its leave-one-out contribution at
**+0.000115**, paired sd 0.000015, 5/5 splits, by fitting the combiner on half of
fold 0's rows and scoring the other half. That was one fold under a split-half
protocol. This is five folds under the fold-wise protocol, so the two are measured
differently and are not required to agree exactly.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")


train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 25's eighteen in row 25's order, then CatBoost. The twelve raw-feature models
# were run before test vectors were being saved as .npy, so their test predictions
# come from the submission csvs, which is what those files are for.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("catboost_te", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
]
NEW = "catboost_te"


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP18 = [i for i, n in enumerate(names) if n != NEW]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:12} {cv:.6f}" + ("   <- new" if n == NEW else ""))


19 members, oof (691369, 19), test (296302, 19)
member CV:


  te42         0.966782


  te2024       0.966771


  te7          0.966729


  te2025       0.966743


...

  bag7         0.963445


  bag2025      0.963337


  bag13        0.963483


  neural       0.939169


  catboost_te  0.966915   <- new


In [3]:
# The fold loop, run twice over the identical folds: once on row 25's eighteen and
# once with CatBoost added. Refitting row 25 here rather than quoting its ledger
# number is what makes the difference paired.
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


per18, test18, _ = run(KEEP18)
per19, test19, coefs = run(list(range(len(names))))

print(f"{'':22} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("18 members, row 25", per18), ("19 members, this run", per19)):
    print(f"{lbl:22} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"18-member CV {per18.mean():.6f} +/- {per18.std():.6f}"
      f"   (row 25 recorded 0.967650 +/- 0.000437, "
      f"diff {per18.mean() - 0.967650:+.2e})")
print(f"19-member CV {per19.mean():.6f} +/- {per19.std():.6f}")


                          fold 0    fold 1    fold 2    fold 3    fold 4
18 members, row 25      0.967003  0.967801  0.967974  0.968179  0.967292
19 members, this run    0.967118  0.967904  0.968057  0.968282  0.967391

18-member CV 0.967650 +/- 0.000437   (row 25 recorded 0.967650 +/- 0.000437, diff -1.23e-07)
19-member CV 0.967750 +/- 0.000431


In [4]:
# Paired, on identical folds. The right test when two models share folds is the
# spread of the per-fold DIFFERENCES and how many folds it wins, not the fold spread,
# which is common to both and cancels.
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:36} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


base17 = np.array([roc_auc_score(y[folds == f], Poof["te42"][folds == f])
                   for f in range(5)])
basecat = np.array([roc_auc_score(y[folds == f], Poof[NEW][folds == f])
                    for f in range(5)])

d_new = paired(per19, per18, "19 vs 18 members (row 25)")
paired(per19, base17, "19-member vs te42 alone (row 17)")
paired(per19, basecat, "19-member vs catboost alone (row 26)")
print()
print("The probe in 17 put CatBoost's leave-one-out contribution at +0.000115 by a")
print("split-half fit on fold 0's rows. The first line above is the same quantity,")
print("measured fold-wise on all five folds.")


19 vs 18 members (row 25)            +0.000100  sd 0.000012  5/5  t(4)=19.18
     per fold: +0.000115  +0.000103  +0.000082  +0.000103  +0.000099
19-member vs te42 alone (row 17)     +0.000968  sd 0.000059  5/5  t(4)=36.62
     per fold: +0.001008  +0.000929  +0.000883  +0.001019  +0.001001
19-member vs catboost alone (row 26) +0.000836  sd 0.000028  5/5  t(4)=66.35
     per fold: +0.000818  +0.000828  +0.000869  +0.000802  +0.000860

The probe in 17 put CatBoost's leave-one-out contribution at +0.000115 by a
split-half fit on fold 0's rows. The first line above is the same quantity,
measured fold-wise on all five folds.


In [5]:
# Coefficient stability. Five fits on 80% overlapping data should agree closely; if
# they did not, the combiner would be fitting fold noise and its weights would not
# mean anything. The last column is what row 25 gave the same member without
# CatBoost in the set, so the shift is readable.
ROW25 = {"te42": 0.1692, "te13": 0.1653, "te2024": 0.1648, "te2025": 0.1553,
         "te7": 0.1362, "lr003": 0.1271, "neural": 0.1178, "lr005": 0.1065,
         "bag42": 0.0932, "bag7": 0.0893, "bag13": 0.0844, "bag2025": 0.0687,
         "bag2024": 0.0197, "trees1000": -0.0076, "trees2000": -0.0126,
         "lr010": -0.0157, "trees300": -0.1349, "anchor": -0.3630}
print(f"{'member':12} {'mean':>9} {'sd across folds':>17}   {'row 25':>8}")
for i in np.argsort(-coefs.mean(axis=0)):
    was = ROW25.get(names[i])
    print(f"{names[i]:12} {coefs[:, i].mean():>+9.4f} {coefs[:, i].std():>17.4f}   "
          + ("     new" if was is None else f"{was:>+8.4f}"))
print(f"\nlargest fold-to-fold sd: {coefs.std(axis=0).max():.4f}")


member            mean   sd across folds     row 25
catboost_te    +0.3512            0.0044        new
neural         +0.1169            0.0016    +0.1178
lr003          +0.1138            0.0126    +0.1271
te13           +0.1015            0.0062    +0.1653
te42           +0.1007            0.0070    +0.1692
te2024         +0.0996            0.0116    +0.1648
lr005          +0.0984            0.0097    +0.1065
te2025         +0.0953            0.0111    +0.1553
bag42          +0.0878            0.0112    +0.0932
bag7           +0.0843            0.0162    +0.0893
bag13          +0.0804            0.0073    +0.0844
te7            +0.0798            0.0129    +0.1362
bag2025        +0.0626            0.0044    +0.0687
bag2024        +0.0172            0.0357    +0.0197
trees1000      +0.0039            0.0094    -0.0076
lr010          -0.0032            0.0088    -0.0157
trees2000      -0.0087            0.0019    -0.0126
trees300       -0.1451            0.0097    -0.1349
anchor      

In [6]:
# Submission: the average of the five fold combiners, matching what every other
# submission in this repo does with its five fold models.
pred = test19.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

# Row 25 was NOT submitted because its test ordering correlated with the submitted
# row 24 at Spearman 0.9999995, so a slot would have re-measured an ordering the
# leaderboard cannot resolve. The same question has to be asked again here, because
# it is the one that decides whether this costs a submission.
prev = pd.read_csv(S / "stack_logit_18.csv")
assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
tau24 = pd.Series(pred).corr(pd.Series(prev["addicted_label"].to_numpy()),
                             method="spearman")
tau25 = pd.Series(pred).corr(pd.Series(test18.mean(axis=0)), method="spearman")
print(f"spearman vs the submitted row 24 : {tau24:.7f}")
print(f"spearman vs row 25, 18 members   : {tau25:.7f}")
print("Row 25 sat at 0.9999995 against row 24 and was held back on that basis.")

out = S / "stack_oof_19.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows, "
      f"range [{prob.min():.4f}, {prob.max():.4f}]")
print(f"ledger: CV {per19.mean():.6f} +/- {per19.std():.6f}, "
      f"vs row 25 {d_new.mean():+.6f} ({(d_new > 0).sum()}/5, "
      f"sd {d_new.std(ddof=1):.6f})")


spearman vs the submitted row 24 : 0.9994618
spearman vs row 25, 18 members   : 0.9994576
Row 25 sat at 0.9999995 against row 24 and was held back on that basis.



wrote stack_oof_19.csv, 296,302 rows, range [0.0000, 1.0000]
ledger: CV 0.967750 +/- 0.000431, vs row 25 +0.000100 (5/5, sd 0.000012)
